In [1]:
import os
import sys
import numpy as np
cwd = os.getcwd()
src_dir = os.path.join(cwd, '..', 'src')
sys.path.append(src_dir)
from dqc.dqc import DQCPipeline
import seaborn as sns
from etl.etl import ETL
from plotter.plotter import Plotter

# 1.Data Extraction
The data is being downloaded via *opendatasets*. *Opendatasets* uses Kaggle API when downloading datasets from Kaggle, therefore it requires your Kaggle Username and Kaggle API key. They can be found in kaggle.json file. In case you don't have kaggle.json, here are steps to get it:<br>
1. Kaggle.com -> Settings -> API -> Create New Token
2. Use "username" value for "Your Kaggle username" and "key" value for "Your Kaggle Key"

In [2]:
etl = ETL()
df = etl.extract(url="https://www.kaggle.com/competitions/competitive-data-science-predict-future-sales")

Skipping, found downloaded files in ".\competitive-data-science-predict-future-sales" (use force=True to force download)


INFO:decorators.exec_time:Function extract took 1.8545 seconds to execute


For my personal comfort `extract` method uses `merge_data` from `dataloader.dataloader` to merge separate tables into one dataframe

# 2.DQC Report
`DQCPipeline` class uses `render_report` function to generate reports on data. `report` is presented as a dictionary of {data check name:pandas DataFrame}. `report` keys include:
- *missing_values* -> returns the percentage of missing values in each column
- *unique_values* -> returns the amount of unique values in each column
- *outliers* -> returns the amount of outliers (based on `IsolationForest`)
- *duplicates* -> returns the number of fully duplicated rows in the table
- *statistics* -> returns table statistical description
- *inconsistencies* -> cheks data for inconsistance (for now only for negative values) 

In [3]:
dqc = DQCPipeline()
report = dqc.render_report(df)

INFO:dqc.dqc:Missing values detection completed successfully - 2025-08-22 15:31:31.299073
INFO:decorators.exec_time:Function _detect_missing_values took 0.2724 seconds to execute
INFO:dqc.dqc:Unique values detected successfully - 2025-08-22 15:31:32.227448
INFO:decorators.exec_time:Function _detect_unique_values took 0.9297 seconds to execute
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    4.4s finished
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    4.2s
[Parallel(n_jobs=1)]: Done 199 tasks      | elapsed:   17.0s
[Parallel(n_jobs=1)]: Done 200 out of 200 | elapsed:   17.1s finished
INFO:dqc.dqc:Outliers detected successfully - 2025-08-22 15:31:54.031830
INFO:decorators.exec_time:Function _detect_outliers took 21.8066 seconds to execute
INFO:dqc.dqc:Duplicates detectes successfully - 2025-08-22 15:31:55.519415
INFO:decorators.exec_time:Function _detect_duplicates took 1.4844 seconds to execute
INFO:dqc.dqc:Statistics calculated successfully - 2025-08-22 15:31:55.521417

# 2.1 Missing Values Report


In [4]:
report["missing_values"]

,missing_values_percentage
date,0.0
date_block_num,0.0
shop_id,0.0
item_id,0.0
item_price,0.0
item_cnt_day,0.0
item_name,0.0
item_category_id,0.0
item_category_name,0.0
shop_name,0.0


- As it can be seen above, there are luckily no missing values in the whole table

# 2.2 Unique Values Report

In [5]:
report["unique_values"]

,unique_values
date,1034
date_block_num,34
shop_id,60
item_id,21807
item_price,19993
item_cnt_day,198
item_name,21807
item_category_id,84
item_category_name,84
shop_name,60


# 2.3 Outliers Report
The main idea of catching outliers was to use machine learning algorithm, not pure statistics. That's why `IsolationForest` was used (nevertheless `LOF` was tested as well). The main problem at first point is that the data contains *id* features, where values can be strangely distributed. Therefore a decision was made to train `IsolationForest` basing only on *item_price* and *item_cnt_day*, which resulted in nearly 7% fewer outliers (may be even mistakenly considered by pure statistics)

In [6]:
report["outliers"]

,outliers_total
outliers,277184


The total percentage of outliers is 9.44% which can be considered as quite high. I can only suppose that the reason for this is some extraordinary items with extremely high/low price or items often/rare sold.

<i>May be it makes sence to try scaling and detect outliers once again, it might perform much better</i>

# 2.4 Duplicates Report

In [7]:
report["duplicates"]

,duplicated_rows
duplicates,6


Only 6 fully duplicated rows which is not that many. So handling duplicates these rows can be deleted with no worries.

# 2.5 Statisctics Report
Provides statistical gatherings including custom percentiles (1%, 25%, 50%, 75%, 99% are set by default)

In [8]:
report["statistics"]

,date_block_num,shop_id,item_id,item_price,item_cnt_day,item_category_id
count,2.935849e+06,2.935849e+06,2.935849e+06,2.935849e+06,2.935849e+06,2.935849e+06
mean,1.456991e+01,3.300173e+01,1.019723e+04,8.908532e+02,1.242641e+00,4.000138e+01
std,9.422988e+00,1.622697e+01,6.324297e+03,1.729800e+03,2.618834e+00,1.710076e+01
min,0.000000e+00,0.000000e+00,0.000000e+00,-1.000000e+00,-2.200000e+01,0.000000e+00
1%,0.000000e+00,2.000000e+00,5.560000e+02,5.000000e+00,1.000000e+00,3.000000e+00
25%,7.000000e+00,2.200000e+01,4.476000e+03,2.490000e+02,1.000000e+00,2.800000e+01
50%,1.400000e+01,3.100000e+01,9.343000e+03,3.990000e+02,1.000000e+00,4.000000e+01
75%,2.300000e+01,4.700000e+01,1.568400e+04,9.990000e+02,1.000000e+00,5.500000e+01
99%,3.300000e+01,5.900000e+01,2.197900e+04,5.999000e+03,5.000000e+00,7.600000e+01
max,3.300000e+01,5.900000e+01,2.216900e+04,3.079800e+05,2.169000e+03,8.300000e+01


# 2.6 Inconsistancies Report
For now *Inconsostancies Report* provides information only about negative values

In [9]:
report["inconsistancies"]

,negative_values
date_block_num,0
shop_id,0
item_id,0
item_price,1
item_cnt_day,7356
item_category_id,0


1 negative value in *item_price* feature which is probably a mistake and 7356 negative values in *item_cnt_day* which is a bit more complicated. There are 2 possible options:
1. A mistake
2. A refund\
Due to data ambigiosity and complexity it's hard to find out whether negative *item_cnt_day* value is a return or a mistake. Here's an example of probably refunded item:


In [10]:
df[1:3]

,date,date_block_num,shop_id,item_id,item_price,item_cnt_day,item_name,item_category_id,item_category_name,shop_name
1,03.01.2013,0,25,2552,899.0,1.0,DEEP PURPLE The House Of Blue Light LP,58,Музыка - Винил,"Москва ТРК ""Атриум"""
2,05.01.2013,0,25,2552,899.0,-1.0,DEEP PURPLE The House Of Blue Light LP,58,Музыка - Винил,"Москва ТРК ""Атриум"""


Despite everything, a mistaken value and a refund can be in some way considered to be the same entity. Therefore they should probably have to be handled the same way. 

# 3. Data Transfromation

`ETL` class uses `transform` function that is responsible for data transfromation. The default `transform` pipeline includes:
1. `handle_missings` -> imputes missing values based on provided strategy (*mean* for numeric and *most_frequent* for categorical by default)
2. `handle_duplicates` -> deletes fully duplicated rows from the table
3. `handle_negatives` -> translates negative values to zero
4. `process_categoricals` -> translates categorical features' values to lowercase
5. `process_date` -> translated *date* from object to datetime
6. `extend_features` -> creates a new feature *city* (might be extended to create new features)

`transform` method takes 2 arguments:
1. df - pandas DataFrame
2. pipeline - custom user pipeline\
In case you think some of transformations are unneccessary, `transform` takes *pipeline* argument as a `List[str]` consisting of *default_pipeline* keys and applies your preferrable transformations to the table 

In [11]:
df = etl.transform(df)
df

INFO:decorators.exec_time:Function _handle_missing_values took 1.5233 seconds to execute
INFO:etl.etl:handle_missings performed successfully - 2025-08-22 15:31:57.616041
INFO:decorators.exec_time:Function _handle_duplicates took 1.9017 seconds to execute
INFO:etl.etl:handle_duplicates performed successfully - 2025-08-22 15:31:59.519724
INFO:decorators.exec_time:Function _handle_negatives took 0.1726 seconds to execute
INFO:etl.etl:handle_negatives performed successfully - 2025-08-22 15:31:59.692331
C:\study\Practice-DS-1.1\src\etl\etl.py:87: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = df[col].astype(str).str.lower()
C:\study\Practice-DS-1.1\src\etl\etl.py:87: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice fr

,date,date_block_num,shop_id,item_id,item_price,item_cnt_day,item_name,item_category_id,item_category_name,shop_name,city
0,2013-01-02,0.0,59.0,22154.0,999.00,1.0,явление 2012 (bd),37.0,кино - blu-ray,"ярославль тц ""альтаир""",ярославль
1,2013-01-03,0.0,25.0,2552.0,899.00,1.0,deep purple the house of blue light lp,58.0,музыка - винил,"москва трк ""атриум""",москва
2,2013-01-05,0.0,25.0,2552.0,899.00,0.0,deep purple the house of blue light lp,58.0,музыка - винил,"москва трк ""атриум""",москва
3,2013-01-06,0.0,25.0,2554.0,1709.05,1.0,deep purple who do you think we are lp,58.0,музыка - винил,"москва трк ""атриум""",москва
4,2013-01-15,0.0,25.0,2555.0,1099.00,1.0,deep purple 30 very best of 2cd (фирм.),56.0,музыка - cd фирменного производства,"москва трк ""атриум""",москва
...,...,...,...,...,...,...,...,...,...,...,...
2935844,2015-10-10,33.0,25.0,7409.0,299.00,1.0,v/a nu jazz selection (digipack),55.0,музыка - cd локального производства,"москва трк ""атриум""",москва
2935845,2015-10-09,33.0,25.0,7460.0,299.00,1.0,v/a the golden jazz collection 1 2cd,55.0,музыка - cd локального производства,"москва трк ""атриум""",москва
2935846,2015-10-14,33.0,25.0,7459.0,349.00,1.0,v/a the best of the 3 tenors,55.0,музыка - cd локального производства,"москва трк ""атриум""",москва
2935847,2015-10-22,33.0,25.0,7440.0,299.00,1.0,v/a relax collection planet mp3 (mp3-cd) (jewel),57.0,музыка - mp3,"москва трк ""атриум""",москва


# 4. Transformed Data DQC Report
One more report on transformed data to make sure all the transformations have been applied

In [12]:
report = dqc.render_report(df)
for key in report.keys():
    display(report[key])

INFO:dqc.dqc:Missing values detection completed successfully - 2025-08-22 15:32:04.135138
INFO:decorators.exec_time:Function _detect_missing_values took 0.3181 seconds to execute
INFO:dqc.dqc:Unique values detected successfully - 2025-08-22 15:32:05.785320
INFO:decorators.exec_time:Function _detect_unique_values took 1.6502 seconds to execute
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    3.7s finished
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    3.9s
[Parallel(n_jobs=1)]: Done 199 tasks      | elapsed:   16.5s
[Parallel(n_jobs=1)]: Done 200 out of 200 | elapsed:   16.6s finished
INFO:dqc.dqc:Outliers detected successfully - 2025-08-22 15:32:26.524890
INFO:decorators.exec_time:Function _detect_outliers took 20.7416 seconds to execute
INFO:dqc.dqc:Duplicates detectes successfully - 2025-08-22 15:32:28.245781
INFO:decorators.exec_time:Function _detect_duplicates took 1.7179 seconds to execute
INFO:dqc.dqc:Statistics calculated successfully - 2025-08-22 15:32:28.247782

,missing_values_percentage
date,0.0
date_block_num,0.0
shop_id,0.0
item_id,0.0
item_price,0.0
item_cnt_day,0.0
item_name,0.0
item_category_id,0.0
item_category_name,0.0
shop_name,0.0


,unique_values
date,1034
date_block_num,34
shop_id,60
item_id,21807
item_price,19993
item_cnt_day,190
item_name,21749
item_category_id,84
item_category_name,84
shop_name,60


,outliers_total
outliers,306524


,duplicated_rows
duplicates,0


,date_block_num,shop_id,item_id,item_price,item_cnt_day,item_category_id
count,2.935843e+06,2.935843e+06,2.935843e+06,2.935843e+06,2.935843e+06,2.935843e+06
mean,1.456991e+01,3.300171e+01,1.019723e+04,8.908535e+02,1.245210e+00,4.000141e+01
std,9.422992e+00,1.622698e+01,6.324293e+03,1.729801e+03,2.617049e+00,1.710076e+01
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
1%,0.000000e+00,2.000000e+00,5.560000e+02,5.000000e+00,1.000000e+00,3.000000e+00
25%,7.000000e+00,2.200000e+01,4.476000e+03,2.490000e+02,1.000000e+00,2.800000e+01
50%,1.400000e+01,3.100000e+01,9.343000e+03,3.990000e+02,1.000000e+00,4.000000e+01
75%,2.300000e+01,4.700000e+01,1.568400e+04,9.990000e+02,1.000000e+00,5.500000e+01
99%,3.300000e+01,5.900000e+01,2.197900e+04,5.999000e+03,5.000000e+00,7.600000e+01
max,3.300000e+01,5.900000e+01,2.216900e+04,3.079800e+05,2.169000e+03,8.300000e+01


,negative_values
date_block_num,0
shop_id,0
item_id,0
item_price,0
item_cnt_day,0
item_category_id,0
